# Prepare the weekly review

In [1]:
# DAY 26 — WEEKLY REVIEW SETUP
#
# Purpose:
# Locate the project and prepare folders for the weekly scorecard,
# interview drill, self-assessment and next-week plan.

from pathlib import Path
from datetime import date, datetime, timezone
import json
import sys


def locate_project_root(starting_folder: Path) -> Path:
    """
    Locate the project from the current notebook folder.
    """

    for possible_root in [
        starting_folder,
        *starting_folder.parents,
    ]:
        has_git = (
            possible_root / ".git"
        ).exists()

        has_project_folders = (
            (possible_root / "app").exists()
            and (possible_root / "data").exists()
        )

        if has_git or has_project_folders:
            return possible_root

    return starting_folder


PROJECT_ROOT = locate_project_root(
    Path.cwd().resolve()
)

for folder_name in [
    "artifacts",
    "data",
    "notes",
    "notebooks",
    "plans",
    "portfolio",
]:
    (PROJECT_ROOT / folder_name).mkdir(
        parents=True,
        exist_ok=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def save_json(relative_path: str, payload) -> Path:
    """
    Save readable JSON inside the project.
    """

    destination = PROJECT_ROOT / relative_path

    destination.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    return destination


def load_json(relative_path: str):
    """
    Load JSON when it exists; otherwise return None.
    """

    source = PROJECT_ROOT / relative_path

    if not source.exists():
        return None

    return json.loads(
        source.read_text(
            encoding="utf-8"
        )
    )


def nested_get(payload, *keys, default=None):
    """
    Safely retrieve a value from nested dictionaries.
    """

    current_value = payload

    for key in keys:
        if not isinstance(
            current_value,
            dict,
        ):
            return default

        if key not in current_value:
            return default

        current_value = current_value[key]

    return current_value


print("Project root:")
print(PROJECT_ROOT)

print("\nPython version:")
print(sys.version.split()[0])

Project root:
/home/e9b07247-894a-49af-bfb3-922e50af0dcf

Python version:
3.11.17


# Generate the weekly capability scorecard

In [3]:
# DAY 26 — CORRECT PROJECT ROOT RECOVERY

from pathlib import Path
import os
import sys


expected_project_files = [
    Path("app") / "resilient_model_client.py",
    Path("prompts") / "finance_assistant_v2.json",
    Path("app") / "prompt_contract.py",
    Path("app") / "governed_context_builder.py",
    (
        Path("artifacts")
        / "day22_approved_context_package.json"
    ),
    Path("app") / "governance_evaluator.py",
    (
        Path("artifacts")
        / "day23_evaluation_report.json"
    ),
    (
        Path("artifacts")
        / "day24_deployment_readiness.json"
    ),
    (
        Path("artifacts")
        / "day25_portfolio_demo_results.json"
    ),
    (
        Path("audit")
        / "day25_trace_chain.jsonl"
    ),
    (
        Path("portfolio")
        / "day25_demo_walkthrough.md"
    ),
]


# Search the current folder, its parents and the
# Anaconda browser project area.
repository_candidates = [
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]


projects_area = Path(
    "/var/www/filebrowser/.projects"
)

if projects_area.exists():
    repository_candidates.extend(
        projects_area.glob(
            "*/sap-finance-ai-assistant"
        )
    )


valid_repositories = []

for candidate in repository_candidates:
    try:
        candidate = candidate.resolve()

        if (
            candidate.exists()
            and (candidate / ".git").exists()
            and candidate not in valid_repositories
        ):
            valid_repositories.append(candidate)

    except (OSError, RuntimeError):
        continue


if not valid_repositories:
    raise FileNotFoundError(
        "The sap-finance-ai-assistant Git repository "
        "could not be located."
    )


# Select the repository containing the most expected files.
def repository_score(repository_root):
    return sum(
        (repository_root / relative_path).exists()
        for relative_path in expected_project_files
    )


PROJECT_ROOT = max(
    valid_repositories,
    key=repository_score,
)

GIT_ROOT = PROJECT_ROOT

os.chdir(PROJECT_ROOT)


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


found_count = repository_score(PROJECT_ROOT)


print("Correct project root:")
print(PROJECT_ROOT)

print(
    "\nExpected capability files found:",
    f"{found_count}/{len(expected_project_files)}",
)

print("\nFile verification:")

for relative_path in expected_project_files:
    if (
        PROJECT_ROOT / relative_path
    ).exists():
        print("✅", relative_path)
    else:
        print("⚠️", relative_path)


if found_count == 0:
    raise AssertionError(
        "A Git repository was found, but none of the "
        "expected project files exist there."
    )


print(
    "\n✅ Project root corrected. "
    "Rerun the Day 26 scorecard cell now."
)

Correct project root:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant

Expected capability files found: 9/11

File verification:
✅ app/resilient_model_client.py
✅ prompts/finance_assistant_v2.json
✅ app/prompt_contract.py
⚠️ app/governed_context_builder.py
⚠️ artifacts/day22_approved_context_package.json
✅ app/governance_evaluator.py
✅ artifacts/day23_evaluation_report.json
✅ artifacts/day24_deployment_readiness.json
✅ artifacts/day25_portfolio_demo_results.json
✅ audit/day25_trace_chain.jsonl
✅ portfolio/day25_demo_walkthrough.md

✅ Project root corrected. Rerun the Day 26 scorecard cell now.


In [12]:
# DAY 26 — CAPABILITY SCORECARD
#
# Purpose:
# Map the week's engineering capabilities to concrete evidence files.
#
# Missing evidence becomes a visible backlog item instead of causing
# an unclear FileNotFoundError.

capability_definitions = [
    {
        "capability": "Python and model-provider APIs",
        "evidence_candidates": [
            "app/resilient_model_client.py",
            "app/btp_service.py",
        ],
        "minimum_required": 2,
    },
    {
        "capability": "Prompt engineering and output contracts",
        "evidence_candidates": [
            "prompts/finance_assistant_v2.json",
            "app/prompt_contract.py",
        ],
        "minimum_required": 2,
    },
    {
        "capability": "Authorized RAG and enterprise context",
        "evidence_candidates": [
            "app/governed_context_builder.py",
            "artifacts/day22_approved_context_package.json",
        ],
        "minimum_required": 2,
    },
    {
        "capability": "Evaluation, security and governance",
        "evidence_candidates": [
            "app/governance_evaluator.py",
            "artifacts/day23_evaluation_report.json",
        ],
        "minimum_required": 2,
    },
    {
        "capability": "SAP BTP deployment readiness",
        "evidence_candidates": [
            "deployment/manifest.yml",
            "artifacts/day24_deployment_readiness.json",
        ],
        "minimum_required": 2,
    },
    {
        "capability": "Portfolio demonstration and observability",
        "evidence_candidates": [
            "artifacts/day25_portfolio_demo_results.json",
            "audit/day25_trace_chain.jsonl",
            "portfolio/day25_demo_walkthrough.md",
        ],
        "minimum_required": 3,
    },
]


capability_results = []

for definition in capability_definitions:
    found_evidence = [
        relative_path
        for relative_path in definition[
            "evidence_candidates"
        ]
        if (
            PROJECT_ROOT / relative_path
        ).exists()
    ]

    missing_evidence = [
        relative_path
        for relative_path in definition[
            "evidence_candidates"
        ]
        if relative_path not in found_evidence
    ]

    capability_complete = (
        len(found_evidence)
        >= definition["minimum_required"]
    )

    capability_results.append(
        {
            "capability": definition[
                "capability"
            ],
            "status": (
                "COMPLETE"
                if capability_complete
                else "GAP"
            ),
            "found_evidence": found_evidence,
            "missing_evidence": missing_evidence,
            "minimum_required": definition[
                "minimum_required"
            ],
        }
    )


day23_report = load_json(
    "artifacts/day23_evaluation_report.json"
)

day23_decision = load_json(
    "governance/day23_release_decision.json"
)

day24_smoke = load_json(
    "artifacts/day24_btp_smoke_test.json"
)

day25_report = load_json(
    "artifacts/day25_portfolio_demo_results.json"
)


day24_smoke_results = nested_get(
    day24_smoke,
    "results",
    default={},
)

completed_capabilities = sum(
    result["status"] == "COMPLETE"
    for result in capability_results
)

capability_completion_rate = (
    completed_capabilities
    / len(capability_results)
)


weekly_scorecard = {
    "review_id": "day26-weekly-capability-review",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "days_reviewed": [
        20,
        21,
        22,
        23,
        24,
        25,
    ],
    "capability_count": len(
        capability_results
    ),
    "completed_capabilities": (
        completed_capabilities
    ),
    "capability_completion_rate": round(
        capability_completion_rate,
        4,
    ),
    "capabilities": capability_results,
    "quality_metrics": {
        "hardened_release_gate": nested_get(
            day23_report,
            "candidate_reports",
            "hardened-v2",
            "release_gate",
        ),
        "human_approval_status": nested_get(
            day23_decision,
            "human_approval_status",
        ),
        "deployment_authorized": nested_get(
            day23_decision,
            "deployment_authorized",
        ),
        "day24_smoke_tests_passed": (
            bool(day24_smoke_results)
            and all(
                day24_smoke_results.values()
            )
        ),
        "day25_demo_pass_rate": nested_get(
            day25_report,
            "pass_rate",
        ),
        "day25_trace_chain_valid": nested_get(
            day25_report,
            "trace_chain",
            "valid",
        ),
    },
}


scorecard_path = save_json(
    "artifacts/day26_weekly_scorecard.json",
    weekly_scorecard,
)


print("Weekly capability scorecard:\n")

for result in capability_results:
    icon = (
        "✅"
        if result["status"] == "COMPLETE"
        else "⚠️"
    )

    print(
        f"{icon} {result['capability']}: "
        f"{result['status']}"
    )

    if result["missing_evidence"]:
        for missing_file in result[
            "missing_evidence"
        ]:
            print(
                "   Missing:",
                missing_file,
            )

print(
    "\nCapability completion:",
    f"{capability_completion_rate:.1%}",
)

print("\nScorecard saved:")
print(scorecard_path)

Weekly capability scorecard:

✅ Python and model-provider APIs: COMPLETE
✅ Prompt engineering and output contracts: COMPLETE
✅ Authorized RAG and enterprise context: COMPLETE
✅ Evaluation, security and governance: COMPLETE
✅ SAP BTP deployment readiness: COMPLETE
✅ Portfolio demonstration and observability: COMPLETE

Capability completion: 100.0%

Scorecard saved:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/artifacts/day26_weekly_scorecard.json


# Create the interview drill

In [13]:
# DAY 26 — INTERVIEW QUESTIONS
#
# Purpose:
# Create five questions that test whether you can explain the project
# as an engineer rather than merely describe individual code cells.

interview_drill = [
    {
        "question_id": "Q1",
        "question": (
            "Walk me through the architecture of your governed "
            "SAP Finance AI assistant."
        ),
        "required_points": [
            "API and input validation",
            "role-aware authorization",
            "authorized retrieval",
            "structured output and citations",
            "human approval",
            "audit and evaluation",
        ],
    },
    {
        "question_id": "Q2",
        "question": (
            "How does your design reduce hallucination and "
            "unsupported Finance guidance?"
        ),
        "required_points": [
            "authorized context only",
            "source citations",
            "structured schema",
            "refusal when context is insufficient",
            "evaluation cases",
        ],
    },
    {
        "question_id": "Q3",
        "question": (
            "What happens when a user asks the assistant to "
            "post, approve or reverse a financial transaction?"
        ),
        "required_points": [
            "no autonomous execution",
            "escalation decision",
            "human approval flag",
            "audit event",
            "least privilege",
        ],
    },
    {
        "question_id": "Q4",
        "question": (
            "How do you determine whether a new prompt or model "
            "version can be released?"
        ),
        "required_points": [
            "same evaluation dataset",
            "quality and security scorers",
            "critical failure gate",
            "technical readiness",
            "separate human deployment approval",
        ],
    },
    {
        "question_id": "Q5",
        "question": (
            "How does this project combine SAP and Databricks "
            "capabilities?"
        ),
        "required_points": [
            "SAP BDC governed data and semantics",
            "SAP Generative AI Hub orchestration",
            "SAP BTP deployment",
            "Databricks evaluation",
            "MLflow tracing",
            "Unity Catalog governance",
        ],
    },
]


interview_path = save_json(
    "data/day26_interview_drill.json",
    interview_drill,
)


print("Interview drill:\n")

for item in interview_drill:
    print(
        item["question_id"],
        item["question"],
    )
    print()

Interview drill:

Q1 Walk me through the architecture of your governed SAP Finance AI assistant.

Q2 How does your design reduce hallucination and unsupported Finance guidance?

Q3 What happens when a user asks the assistant to post, approve or reverse a financial transaction?

Q4 How do you determine whether a new prompt or model version can be released?

Q5 How does this project combine SAP and Databricks capabilities?



# Review model answers and record your scores

In [14]:
# DAY 26 — MODEL ANSWERS AND SELF-ASSESSMENT
#
# Purpose:
# Compare your spoken explanations against concise model answers,
# then record an honest confidence score for each answer.

model_answers = {
    "Q1": (
        "The request enters a FastAPI service where its schema and "
        "role are validated. The retrieval layer selects only policy "
        "context authorized for that role and business unit. The "
        "assistant returns a structured answer with policy citations, "
        "refuses unsupported questions and escalates transaction "
        "execution to a human. Evaluation, release gates and audit "
        "traces provide evidence across the lifecycle."
    ),
    "Q2": (
        "I constrain the assistant to authorized retrieved context, "
        "require citations from that context and validate an exact "
        "output schema. When evidence is unavailable, the correct "
        "behavior is refusal rather than generation from general "
        "model knowledge. Regression tests detect unsupported facts, "
        "invented citations and restricted-data leakage."
    ),
    "Q3": (
        "The assistant does not receive authority to execute Finance "
        "transactions. It returns an escalation decision, sets the "
        "human-approval indicator and records the request in the "
        "audit trail. The responsible Finance approver remains the "
        "accountable decision-maker."
    ),
    "Q4": (
        "Every candidate is tested against the same evaluation set. "
        "I score schema compliance, grounding, citation integrity, "
        "access leakage, refusal and approval behavior. Critical "
        "failures block the technical gate. Even after the technical "
        "gate passes, production authorization remains a separate "
        "human governance decision."
    ),
    "Q5": (
        "SAP Business Data Cloud supplies governed enterprise data "
        "and business semantics. SAP Generative AI Hub provides "
        "model orchestration and controls such as grounding, masking "
        "and filtering, while SAP BTP hosts the API. Databricks and "
        "MLflow provide evaluation, tracing, human feedback and "
        "governance through Unity Catalog."
    ),
}


# EDIT THESE VALUES BEFORE RUNNING THIS CELL.
self_scores = {
    "Q1": 4,
    "Q2": 3,
    "Q3": 3,
    "Q4": 5,
    "Q5": 5,
}


scores_complete = all(
    isinstance(score, int)
    and 1 <= score <= 5
    for score in self_scores.values()
)

average_score = (
    round(
        sum(self_scores.values())
        / len(self_scores),
        2,
    )
    if scores_complete
    else None
)


interview_assessment = {
    "assessment_id": (
        "day26-interview-self-assessment"
    ),
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scores": self_scores,
    "scores_complete": scores_complete,
    "average_score": average_score,
}


assessment_path = save_json(
    "artifacts/day26_interview_self_assessment.json",
    interview_assessment,
)


answer_sections = []

for item in interview_drill:
    question_id = item["question_id"]

    answer_sections.append(
        f"""## {question_id}: {item["question"]}

{model_answers[question_id]}

**My score:** {self_scores[question_id]}
"""
    )


answers_text = (
    "# Day 26 — Interview Drill and Model Answers\n\n"
    + "\n".join(answer_sections)
)

answers_path = (
    PROJECT_ROOT
    / "portfolio"
    / "day26_interview_answers.md"
)

answers_path.write_text(
    answers_text,
    encoding="utf-8",
)


if scores_complete:
    print(
        "Interview self-assessment complete."
    )
    print(
        "Average score:",
        average_score,
        "/ 5",
    )
else:
    print(
        "⚠️ Replace every None with a score from 1 to 5, "
        "then run this cell again."
    )

print("\nModel answers saved:")
print(answers_path)

Interview self-assessment complete.
Average score: 4.0 / 5

Model answers saved:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/portfolio/day26_interview_answers.md


In [16]:
# DAY 26 — FIX OUTPUT FOLDERS AND save_json()

from pathlib import Path
import json


PROJECT_ROOT = Path(
    PROJECT_ROOT
).resolve()


# Create the Day 26 output folders.
for folder_name in [
    "artifacts",
    "notes",
    "plans",
]:
    (
        PROJECT_ROOT / folder_name
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


# Replace the earlier save_json function with
# one that automatically creates parent folders.
def save_json(
    relative_path,
    payload,
):
    destination = (
        PROJECT_ROOT
        / Path(relative_path)
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    return destination


print(
    "✅ Day 26 output folders are ready."
)

print(
    "✅ Corrected save_json() is ready."
)

✅ Day 26 output folders are ready.
✅ Corrected save_json() is ready.


# Generate the Day 27–33 plan

In [17]:
# DAY 26 — NEXT-WEEK PREPARATION
#
# Purpose:
# Define the next seven sequential assignments without repeating
# this week's work.

next_week_plan = [
    {
        "day": 27,
        "date": "2026-11-02",
        "theme": "Monday — Python and APIs",
        "objective": (
            "Add async request handling, idempotency keys and "
            "timeout control to the model-provider interface."
        ),
        "portfolio_evidence": (
            "Async provider adapter and concurrency tests"
        ),
    },
    {
        "day": 28,
        "date": "2026-11-03",
        "theme": (
            "Tuesday — SAP Generative AI Hub and prompts"
        ),
        "objective": (
            "Create a mock SAP orchestration adapter covering "
            "templating, masking and content-filter configuration."
        ),
        "portfolio_evidence": (
            "SAP orchestration configuration contract"
        ),
    },
    {
        "day": 29,
        "date": "2026-11-04",
        "theme": (
            "Wednesday — RAG and enterprise data"
        ),
        "objective": (
            "Upgrade keyword retrieval to semantic retrieval while "
            "preserving role and business-unit filters."
        ),
        "portfolio_evidence": (
            "Local embedding index and authorized retrieval tests"
        ),
    },
    {
        "day": 30,
        "date": "2026-11-05",
        "theme": (
            "Thursday — Evaluation, security and governance"
        ),
        "objective": (
            "Measure retrieval precision, recall, citation validity "
            "and access leakage for the semantic retriever."
        ),
        "portfolio_evidence": (
            "RAG quality and security evaluation report"
        ),
    },
    {
        "day": 31,
        "date": "2026-11-06",
        "theme": (
            "Friday — SAP BTP integration and deployment"
        ),
        "objective": (
            "Parse simulated VCAP_SERVICES bindings safely and "
            "test startup failure when required bindings are absent."
        ),
        "portfolio_evidence": (
            "BTP service-binding adapter and configuration tests"
        ),
    },
    {
        "day": 32,
        "date": "2026-11-07",
        "theme": "Saturday — Portfolio project",
        "objective": (
            "Build a simple reviewer interface for Finance answers, "
            "citations and human approval decisions."
        ),
        "portfolio_evidence": (
            "Reviewer UI prototype and screenshots"
        ),
    },
    {
        "day": 33,
        "date": "2026-11-08",
        "theme": (
            "Sunday — Review and interview preparation"
        ),
        "objective": (
            "Demonstrate semantic RAG, configuration controls and "
            "human review in a five-minute interview walkthrough."
        ),
        "portfolio_evidence": (
            "Week-four scorecard and demonstration script"
        ),
    },
]


plan_path = save_json(
    "plans/day27_33_next_week.json",
    next_week_plan,
)


print("Next-week plan:\n")

for plan_item in next_week_plan:
    print(
        f"Day {plan_item['day']} — "
        f"{plan_item['date']} — "
        f"{plan_item['theme']}"
    )
    print(
        " ",
        plan_item["objective"],
    )
    print()

print("Plan saved:")
print(plan_path)

Next-week plan:

Day 27 — 2026-11-02 — Monday — Python and APIs
  Add async request handling, idempotency keys and timeout control to the model-provider interface.

Day 28 — 2026-11-03 — Tuesday — SAP Generative AI Hub and prompts
  Create a mock SAP orchestration adapter covering templating, masking and content-filter configuration.

Day 29 — 2026-11-04 — Wednesday — RAG and enterprise data
  Upgrade keyword retrieval to semantic retrieval while preserving role and business-unit filters.

Day 30 — 2026-11-05 — Thursday — Evaluation, security and governance
  Measure retrieval precision, recall, citation validity and access leakage for the semantic retriever.

Day 31 — 2026-11-06 — Friday — SAP BTP integration and deployment
  Parse simulated VCAP_SERVICES bindings safely and test startup failure when required bindings are absent.

Day 32 — 2026-11-07 — Saturday — Portfolio project
  Build a simple reviewer interface for Finance answers, citations and human approval decisions.

Day 33 

# Create the weekly review document

In [18]:
# DAY 26 — WEEKLY DOCUMENTATION
#
# Purpose:
# Create one GitHub-readable review containing accomplishments,
# evidence gaps, interview readiness and next-week priorities.

capability_rows = "\n".join(
    (
        f"| {result['capability']} "
        f"| {result['status']} "
        f"| {len(result['found_evidence'])} |"
    )
    for result in capability_results
)

gap_items = [
    missing_file
    for result in capability_results
    for missing_file in result[
        "missing_evidence"
    ]
]

gap_text = (
    "\n".join(
        f"- `{missing_file}`"
        for missing_file in gap_items
    )
    if gap_items
    else "- No evidence gaps detected."
)

plan_rows = "\n".join(
    (
        f"| {item['day']} "
        f"| {item['date']} "
        f"| {item['theme']} "
        f"| {item['objective']} |"
    )
    for item in next_week_plan
)


review_text = f"""# Day 26 — Weekly AI Engineering Review

## Weekly result

- Capabilities assessed: {len(capability_results)}
- Capabilities complete: {completed_capabilities}
- Evidence completion: {capability_completion_rate:.1%}
- Hardened release gate: {weekly_scorecard["quality_metrics"]["hardened_release_gate"]}
- Day 24 smoke tests passed: {weekly_scorecard["quality_metrics"]["day24_smoke_tests_passed"]}
- Day 25 demonstration pass rate: {weekly_scorecard["quality_metrics"]["day25_demo_pass_rate"]}
- Day 25 trace chain valid: {weekly_scorecard["quality_metrics"]["day25_trace_chain_valid"]}
- Interview self-score: {average_score}

## Capability matrix

| Capability | Status | Evidence files found |
|---|---|---:|
{capability_rows}

## Evidence backlog

{gap_text}

## Ninety-second interview pitch

I am transitioning from SAP Finance into AI engineering by building a
governed Finance assistant around enterprise controls rather than only
a model call. The assistant validates roles, retrieves authorized
synthetic policy context and produces structured answers with source
citations. Unsupported requests are refused, while requests to execute
financial transactions are escalated for accountable human approval.

I created regression evaluations for grounding, citation integrity,
access leakage and approval behavior. I then wrapped the solution in a
FastAPI service, prepared it for SAP BTP Cloud Foundry and generated an
auditable portfolio demonstration. The design maps to SAP Business Data
Cloud and Generative AI Hub for governed data and orchestration, and to
Databricks MLflow for evaluation, tracing and continuous improvement.

## Next week

| Day | Date | Theme | Objective |
|---:|---|---|---|
{plan_rows}
"""


review_path = (
    PROJECT_ROOT
    / "notes"
    / "day26_weekly_review.md"
)

review_path.write_text(
    review_text,
    encoding="utf-8",
)

print("Weekly review created:")
print(review_path)

Weekly review created:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/notes/day26_weekly_review.md


# One completion check

In [19]:
# DAY 26 — COMPLETION CHECK
#
# Purpose:
# Confirm that the weekly review, interview assessment and next-week
# plan are complete using one combined check.

expected_files = [
    "artifacts/day26_weekly_scorecard.json",
    "data/day26_interview_drill.json",
    "artifacts/day26_interview_self_assessment.json",
    "portfolio/day26_interview_answers.md",
    "plans/day27_33_next_week.json",
    "notes/day26_weekly_review.md",
]

day25_pass_rate = weekly_scorecard[
    "quality_metrics"
]["day25_demo_pass_rate"]

completion_conditions = {
    "all Day 26 files exist": all(
        (PROJECT_ROOT / relative_path).exists()
        for relative_path in expected_files
    ),
    "six capabilities were reviewed": (
        weekly_scorecard[
            "capability_count"
        ]
        == 6
    ),
    "interview scores were entered": (
        scores_complete
    ),
    "five interview questions were prepared": (
        len(interview_drill) == 5
    ),
    "seven next-week sessions were planned": (
        len(next_week_plan) == 7
    ),
    "Day 25 portfolio demonstration passed": (
        day25_pass_rate == 1.0
    ),
    "Day 25 trace chain was valid": (
        weekly_scorecard[
            "quality_metrics"
        ]["day25_trace_chain_valid"]
        is True
    ),
    "weekly review document exists": (
        review_path.exists()
    ),
}

print("Day 26 completion check:\n")

for condition_name, condition_passed in completion_conditions.items():
    icon = "✅" if condition_passed else "❌"
    print(f"{icon} {condition_name}")

assert all(completion_conditions.values()), (
    "Day 26 is not complete. Review the failed condition above."
)

print(
    "\n✅ DAY 26 COMPLETE — the week is reviewed, "
    "your interview explanations are scored and "
    "Days 27–33 are prepared."
)

Day 26 completion check:

✅ all Day 26 files exist
✅ six capabilities were reviewed
✅ interview scores were entered
✅ five interview questions were prepared
✅ seven next-week sessions were planned
✅ Day 25 portfolio demonstration passed
✅ Day 25 trace chain was valid
✅ weekly review document exists

✅ DAY 26 COMPLETE — the week is reviewed, your interview explanations are scored and Days 27–33 are prepared.
